# PCA and SVD: The Linear Algebra You Actually Need

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/unsupervised/pca_svd_relationship.ipynb)

Derive PCA two ways: via eigen decomposition and via SVD. Prove they give identical results, build geometric intuition, and compress images with truncated SVD.

**Blog post:** [PCA and SVD: The Linear Algebra You Actually Need](https://sesen.ai/blog/pca-svd-linear-algebra-relationship)

**Key references:**
- Hotelling, H. (1933). Analysis of a complex of statistical variables into principal components. *J. Educational Psychology*, 24(6), 417-441.
- Golub, G. & Van Loan, C. (1996). *Matrix Computations* (3rd ed.). Johns Hopkins University Press.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, fetch_olivetti_faces
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

## 1. PCA via Eigen Decomposition (Iris Dataset)

In [ ]:
# Load and standardise
iris = load_iris()
X = StandardScaler().fit_transform(iris.data)  # 150 x 4
labels = iris.target
feature_names = iris.feature_names
species = iris.target_names

print(f"Data shape: {X.shape}")
print(f"Features: {feature_names}")
print(f"Species: {list(species)}")

In [ ]:
# Eigen decomposition of the covariance matrix
cov_matrix = np.cov(X, rowvar=False)  # 4 x 4
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Sort by decreasing eigenvalue
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Variance explained
var_explained = eigenvalues / eigenvalues.sum()
cum_var = np.cumsum(var_explained)

print("Eigenvalues:", np.round(eigenvalues, 4))
print("Variance explained:", [f"{v:.1%}" for v in var_explained])
print("Cumulative:", [f"{v:.1%}" for v in cum_var])

In [ ]:
# Project onto first two PCs and plot
scores = X @ eigenvectors[:, :2]

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#E53935', '#2196F3', '#4CAF50']
for i, name in enumerate(species):
    mask = labels == i
    ax.scatter(scores[mask, 0], scores[mask, 1], c=colors[i],
               label=name.capitalize(), alpha=0.7, s=60, edgecolors='white', linewidth=0.5)
ax.set_xlabel(f'PC1 ({var_explained[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({var_explained[1]:.1%} variance)', fontsize=12)
ax.set_title('Iris PCA: Eigen Decomposition', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 2. PCA via SVD

In [ ]:
# SVD of the centred data matrix
U, S, Vt = np.linalg.svd(X, full_matrices=False)

n = X.shape[0]
eigenvalues_from_svd = S**2 / (n - 1)

print(f"{'':>4} {'Eigen':>12} {'SVD s²/(n-1)':>14} {'Difference':>14}")
for i in range(4):
    diff = abs(eigenvalues[i] - eigenvalues_from_svd[i])
    print(f"PC{i+1} {eigenvalues[i]:>12.4f} {eigenvalues_from_svd[i]:>14.4f} {diff:>14.2e}")

In [ ]:
# Verify eigenvectors match (up to sign)
V_svd = Vt.T
for i in range(4):
    # Align signs
    sign = np.sign(eigenvectors[0, i] * V_svd[0, i])
    diff = np.max(np.abs(eigenvectors[:, i] - sign * V_svd[:, i]))
    print(f"PC{i+1} eigenvector max diff: {diff:.2e}")

print("\nEigenvalues match:", np.allclose(eigenvalues, eigenvalues_from_svd))
print("Eigenvectors match (up to sign):", all(
    np.allclose(np.abs(eigenvectors[:, i]), np.abs(V_svd[:, i])) for i in range(4)
))

## 3. Variance Explained

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Individual variance
axes[0].bar(range(1, 5), var_explained * 100, color='#2196F3', edgecolor='white', linewidth=1.5)
axes[0].set_xlabel('Principal Component', fontsize=12)
axes[0].set_ylabel('Variance Explained (%)', fontsize=12)
axes[0].set_title('Individual Variance', fontsize=13)
axes[0].set_xticks(range(1, 5))
for i, v in enumerate(var_explained):
    axes[0].text(i + 1, v * 100 + 1.5, f'{v:.1%}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylim(0, 85)
axes[0].grid(True, alpha=0.3, axis='y')

# Cumulative variance
axes[1].plot(range(1, 5), cum_var * 100, 'o-', color='#FF9800', linewidth=2, markersize=10)
axes[1].axhline(y=95, color='gray', linestyle='--', alpha=0.5, label='95% threshold')
axes[1].set_xlabel('Number of Components', fontsize=12)
axes[1].set_ylabel('Cumulative Variance (%)', fontsize=12)
axes[1].set_title('Cumulative Variance', fontsize=13)
axes[1].set_xticks(range(1, 5))
axes[1].set_ylim(65, 102)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 4. The Geometry of SVD

In [ ]:
# Create a simple 2x2 matrix to visualise SVD geometry
A = np.array([[3, 1], [1, 2]], dtype=float)
U2, S2, Vt2 = np.linalg.svd(A)

# Unit circle
theta = np.linspace(0, 2*np.pi, 100)
circle = np.vstack([np.cos(theta), np.sin(theta)])

# Apply each step
step1 = Vt2 @ circle          # Rotate by V^T
step2 = np.diag(S2) @ step1   # Scale by Sigma
step3 = U2 @ step2            # Rotate by U

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Unit Circle', r'After $V^T$ (rotate)', r'After $\Sigma$ (scale)', r'After $U$ (rotate)']
data = [circle, step1, step2, step3]
colors_geom = ['#2196F3', '#4CAF50', '#FF9800', '#E53935']

for ax, title, d, c in zip(axes, titles, data, colors_geom):
    ax.plot(d[0], d[1], color=c, linewidth=2.5)
    ax.plot(d[0, 0], d[1, 0], 'o', color='black', markersize=6)  # reference point
    ax.set_title(title, fontsize=12)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.axvline(x=0, color='gray', linewidth=0.5)
    lim = max(abs(d).max() * 1.3, 1.5)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)

fig.suptitle('SVD Geometry: Every Matrix = Rotate → Scale → Rotate', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

print(f"Singular values: {S2[0]:.3f}, {S2[1]:.3f}")
print(f"These are the semi-axis lengths of the ellipse.")

## 5. Image Compression with Truncated SVD

In [ ]:
# Load a face image
faces = fetch_olivetti_faces()
face = faces.images[0]  # 64 x 64
print(f"Image shape: {face.shape}, pixels: {face.size}")

# SVD of the face
U_face, S_face, Vt_face = np.linalg.svd(face, full_matrices=False)

def reconstruct(U, S, Vt, k):
    return U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]

# Show reconstructions at various ranks
ranks = [1, 2, 5, 10, 20, 64]
fig, axes = plt.subplots(1, len(ranks), figsize=(15, 3))
for ax, k in zip(axes, ranks):
    recon = reconstruct(U_face, S_face, Vt_face, k)
    ax.imshow(recon, cmap='gray', vmin=0, vmax=1)
    storage = k * (64 + 1 + 64)  # k columns of U + k singular values + k rows of Vt
    ratio = storage / face.size * 100
    ax.set_title(f'Rank {k}\n{ratio:.0f}% storage', fontsize=10)
    ax.axis('off')
fig.suptitle('Face Reconstruction with Truncated SVD', fontsize=13, y=1.05)
fig.tight_layout()
plt.show()

In [ ]:
# Singular value spectrum
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(S_face) + 1), S_face, color='#2196F3', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Component', fontsize=12)
ax.set_ylabel('Singular Value', fontsize=12)
ax.set_title('Singular Value Spectrum (Face Image)', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')
fig.tight_layout()
plt.show()

# Cumulative energy
energy = np.cumsum(S_face**2) / np.sum(S_face**2)
for k in [5, 10, 20, 30]:
    print(f"Rank {k:2d}: {energy[k-1]:.1%} of total energy")

## 6. Verify with scikit-learn

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=4)
scores_sklearn = pca.fit_transform(X)

print("sklearn explained_variance_:", np.round(pca.explained_variance_, 4))
print("Our eigenvalues:            ", np.round(eigenvalues, 4))
print("Match:", np.allclose(pca.explained_variance_, eigenvalues))

print("\nsklearn components_ (first 2):")
print(np.round(pca.components_[:2], 4))
print("\nOur eigenvectors (first 2):")
print(np.round(eigenvectors[:, :2].T, 4))

## Exercises

1. **Try a larger dataset.** Replace Iris with the Wine or Breast Cancer dataset from scikit-learn. How many components do you need to capture 95% of the variance?

2. **Compress a colour image.** Apply SVD separately to the R, G, and B channels. How does the compression ratio compare to greyscale?

3. **Denoise with truncated SVD.** Add Gaussian noise to a face image, then reconstruct at various ranks. Find the rank that minimises reconstruction error on a held-out clean image.

4. **Compare with scikit-learn.** Run `sklearn.decomposition.PCA` on the Iris data and verify that its `components_` match our eigenvectors and its `explained_variance_` matches our eigenvalues.